## Prepare atlas for upload

This notebook prepares the anndata object containing your atlas data and metadata by removing any data that is not needed for the mapping process. This is a necessary step to make sure that atlas adheres to the resource constraints of ArchMap.

Please make sure you change the names of the input paths in the cell below. Both the atlas h5ad file and the model.pt file should be saved in the atlas_path directory. If you are uploading an atlas integrated using scPoli and have 3 files as your output (attr.pkl, model_params.pt, var_names.csv), please combine these files into one file using the notebook [here]("https://drive.google.com/file/d/1f-RH-4bU4UeTu5HVTB1e1ySYwdCyFf3F/view?usp=drive_link").

After running all the cells, this notebook will output a "cleaned" h5ad file that can be uploaded to ArchMap.

In [ ]:
# Please change the input paths to match you directory and file names
atlas_filename = "test_data/model_hlca_scanvi/adata.h5ad"
atlas_path = "test_data/model_hlca_scanvi/"

In [ ]:
# Run this cell if you do not have these packages installed already
!pip install scanpy

In [ ]:
import scanpy as sc
import torch
import pandas as pd
import numpy as np



def clean_data(adatafile_local, modelpath_local):


    adata = sc.read(f"{adatafile_local}")

    model = torch.load(f"{modelpath_local}/model.pt", map_location="cpu", weights_only=False)

    # check that adata is not already minified
    if (adata.X is None or not adata.X.sum()>0):
        raise ValueError(f"The uploaded h5ad file does not have count data saved in the .X attribute. Please reupload your atlas with count data you used for integration (either raw or log-normalized) in .X.")


    adata=adata[:,pd.Series(model["var_names"]).values]

    del adata.uns
    del adata.obsm
    del adata.obsp
    del adata.varm
    del adata.layers
    del adata.varp
    del adata.raw


    adata.write(f"adata_cleaned.h5ad")


In [ ]:
clean_data(atlas_filename, atlas_path)